In [ ]:
import gurobipy as gp
from gurobipy import GRB
import data
from PriceProcess import price_model
from WindProcess import wind_model
import matplotlib.pyplot as plt

# Helper functions

## Plotting function

In [ ]:
def plot_results(results, fixed_data):
    T = fixed_data['num_timeslots']  
    plt.figure(figsize=(14, 10))
    print(T)

    plt.subplot(8, 1, 1)
    plt.plot(range(T), results['power_wind'], label="Wind Power", color="blue")
    plt.ylabel("Wind Power")
    plt.legend()

    plt.subplot(8, 1, 2)
    plt.plot(range(T), fixed_data['demand_schedule'], label="Demand Schedule", color="orange")
    plt.ylabel("Demand")
    plt.legend()

    plt.subplot(8, 1, 3)
    plt.step(range(T), results['electrolyzer_status'], label="Electrolyzer Status", color="red", where="post")
    plt.ylabel("El. Status")
    plt.legend()


    plt.subplot(8, 1, 4)
    plt.plot(range(T), results['hydrogen_tank'], label="Hydrogen Level", color="green")
    plt.ylabel("Hydr. Level")
    plt.legend()


    plt.subplot(8, 1, 5)
    plt.plot(range(T), results['power_p2h'], label="p2h", color="orange")
    plt.ylabel("p2h")
    plt.legend()


    plt.subplot(8, 1, 6)
    plt.plot(range(T), results['hydrogen_h2p'], label="h2p", color="blue")
    plt.ylabel("h2p")
    plt.legend()


    plt.subplot(8, 1, 7)
    plt.plot(range(T), results['power_grid'], label="Grid Power", color="green")
    plt.ylabel("Grid Power")
    plt.legend()

    plt.subplot(8, 1, 8)
    plt.plot(range(T), results['el_price'], label="price", color="red")
    plt.ylabel("Price")
    plt.xlabel("Time")
    plt.legend()



    plt.tight_layout()
    plt.show()


## Deterministic model

In [ ]:
def get_estimates(T, current_wind, previous_wind, current_price, previous_price, fixed_data):
    wind = [0 for _ in range(T)]
    for t in range(T):
        wind[t] = wind_model(current_wind, previous_wind, fixed_data)
        previous_wind = current_wind
        current_wind = wind[t]

    # get prices
    el_prices = [0 for _ in range(T)]
    for t in range(T):
        el_prices[t] = price_model(current_price, previous_price, wind[t], fixed_data)
        previous_price = current_price
        current_price = el_prices[t]

    return wind, el_prices

In [ ]:
def hindsight_model(fixed_data, el_prices, wind, T, initial_hydrogen, initial_status, outputFlag=0):

    # 2. Create Model
    m = gp.Model("Energy Hub Optimization")

    # 3. Decision Variables
    P_grid = m.addVars(T, vtype=GRB.CONTINUOUS, lb=0, name="grid_power")
    P_p2h = m.addVars(T, vtype=GRB.CONTINUOUS, lb=0, ub=10, name="power_to_hydrogen")
    q_h2p = m.addVars(T, vtype=GRB.CONTINUOUS, lb=0, ub=10, name="hydrogen_to_power")
    U_on = m.addVars(T, vtype=GRB.BINARY, name="ElectrolyzerOn")
    U_off = m.addVars(T, vtype=GRB.BINARY, name="ElectrolyzerOff")
    e = m.addVars(T, vtype=GRB.BINARY, name="electrolyzer_status")
    h = m.addVars(T, vtype=GRB.CONTINUOUS, lb=0, ub=fixed_data['hydrogen_capacity'], name="hydrogen_storage_level")

    # 4. Objective: Minimize (Fixed Costs + Shipping Costs)
    obj = gp.quicksum(P_grid[t] * el_prices[t] + e[t] * fixed_data['electrolyzer_cost'] for t in range(T)) 
    m.setObjective(obj, GRB.MINIMIZE)

    # 5. Constraints
    # Electroyzer dynamics
    m.addConstrs((e[t] == e[t-1] + U_on[t-1] - U_off[t-1]   for t in range(T) if t > 0), name="ElectrolyzerDynamics")
    m.addConstr((e[0] == initial_status), name="ElectrolyzerInit")

    # Hydrogen storage
    m.addConstrs((h[t] == h[t-1] + fixed_data["conversion_p2h"]*P_p2h[t-1] - q_h2p[t-1] for t in range(T) if t > 0), name="HydrogenConversation")
    m.addConstr((h[0] == initial_hydrogen), name="HydrogenInit")
    m.addConstrs((h[t] <= fixed_data["hydrogen_capacity"] for t in range(T)), name="HydrogenMaxLevel")

    # Grid power constraints
    m.addConstrs((P_grid[t] >= fixed_data["demand_schedule"][t] + P_p2h[t] - wind[t] - fixed_data["conversion_h2p"]*q_h2p[t] for t in range(T)), name="Powerconversation")
    m.addConstrs((P_grid[t] >= 0 for t in range(T)), name="non-linear")

    # Operational Constraints
    m.addConstrs((U_on[t] + U_off[t] <= 1 for t in range(T)), name="Operational1")
    m.addConstrs((U_off[t] <= e[t] for t in range(T)), name="Operational2")
    m.addConstrs((U_on[t] <= 1 - e[t] for t in range(T)), name="Operational3")

    # power Conversion Limits
    m.addConstrs((P_p2h[t] <= e[t]*fixed_data["p2h_max_rate"] for t in range(T)), name="maxP2H")
    m.addConstrs((q_h2p[t] <= h[t] for t in range(T)), name="maxTank")
    m.addConstrs((q_h2p[t] <= fixed_data["h2p_max_rate"] for t in range(T)), name="maxH2P")


    # 6. Optimize
    m.setParam("OutputFlag", outputFlag)
    m.optimize()
    results = {
        'hydrogen_tank': [h[t].X for t in range(T)],
        'power_p2h': [P_p2h[t].X for t in range(T)],
        'hydrogen_h2p': [q_h2p[t].X for t in range(T)],
        'power_grid': [P_grid[t].X for t in range(T)],
        'electrolyzer_status': [e[t].X for t in range(T)],
        'activate_electrolyzer': [U_on[t].X for t in range(T)],
        'deactivate_electrolyzer': [U_off[t].X for t in range(T)],
        'el_price': el_prices,
        'power_wind': wind,
    }
    return results


## Policies

In [ ]:
def adp_policy(state, fixed_data, t):
    return None

In [ ]:
def multi_stage_policy(state, fixed_data, t, stages, num_variables):
    return None

In [ ]:
def two_stage_policy(state, fixed_data, t):
    """Two‑stage stochastic policy.

    We sample a small number of possible realizations for the next
    time step (wind and price) and solve a two‑period deterministic
    lookahead for each scenario.  The first‑stage decision is the
    (possibly fractional) average over all scenarios.  Binary
    actions (electrolyzer on/off) are rounded.
    """
    # number of scenarios to approximate the expectation
    num_scenarios = 5
    # accumulate decisions
    agg = {'power_p2h': 0.0,
           'hydrogen_h2p': 0.0,
           'power_grid': 0.0,
           'activate_electrolyzer': 0.0,
           'deactivate_electrolyzer': 0.0}

    for _ in range(num_scenarios):
        # forecast two periods ahead using the stochastic models
        wind, el_prices = get_estimates(
            2,
            state['power_wind'],
            state['power_wind_previous'],
            state['el_price'],
            state['el_price_previous'],
            fixed_data,
        )
        # solve the two‑period deterministic optimization
        results = hindsight_model(
            fixed_data,
            el_prices,
            wind,
            2,
            state['hydrogen_tank'],
            state['electrolyzer_status'],
            outputFlag=0,
        )
        # accumulate first‑stage decision
        agg['power_p2h'] += results['power_p2h'][0]
        agg['hydrogen_h2p'] += results['hydrogen_h2p'][0]
        agg['power_grid'] += results['power_grid'][0]
        agg['activate_electrolyzer'] += results['activate_electrolyzer'][0]
        agg['deactivate_electrolyzer'] += results['deactivate_electrolyzer'][0]

    # average and round logical variables
    for k in ['power_p2h', 'hydrogen_h2p', 'power_grid']:
        agg[k] /= num_scenarios
    agg['activate_electrolyzer'] = agg['activate_electrolyzer'] / num_scenarios >= 0.5
    agg['deactivate_electrolyzer'] = agg['deactivate_electrolyzer'] / num_scenarios >= 0.5

    return agg

In [ ]:
def deterministic_lookahead_policy(state, fixed_data, t, L):
    wind, el_prices = get_estimates(L, state['power_wind'], state['power_wind_previous'], state['el_price'], state['el_price_previous'], fixed_data)
    results = hindsight_model(fixed_data, el_prices, wind, L, state['hydrogen_tank'], state['electrolyzer_status'], outputFlag=0)
    # print(results)
    return {'power_p2h': results['power_p2h'][0],'hydrogen_h2p': results['hydrogen_h2p'][0],'power_grid': results['power_grid'][0],'activate_electrolyzer': results['activate_electrolyzer'][0],'deactivate_electrolyzer': results['deactivate_electrolyzer'][0]}

In [ ]:
def dummy_policy(state, fixed_data, t):
    p_grid = 0
    if state['power_wind'] < fixed_data['demand_schedule'][t]:
        p_grid = fixed_data['demand_schedule'][t] - state['power_wind']
    return {'power_p2h': 0,'hydrogen_h2p': 0,'power_grid': p_grid,'activate_electrolyzer': False,'deactivate_electrolyzer': False}

## MDP Logic

In [ ]:
def apply_policy(state, fixed_data, t, policy='dummy', num_lookaheads=20, num_stages=3):
    if policy == 'dummy':
        return dummy_policy(state, fixed_data, t)
    if policy == 'deterministic_lookahead':
        return deterministic_lookahead_policy(state, fixed_data, t, L=num_lookaheads)
    if policy == 'two_stage':
        return two_stage_policy(state, fixed_data, t)
    if policy == 'multi_stage':
        return multi_stage_policy(state, fixed_data, t, stages=num_stages)
    if policy == 'adp':
        return adp_policy(state, fixed_data, t)
    if policy == 'infeasible':
        return {'power_p2h': 0,'hydrogen_h2p': 0, 'activate_electrolyzer': True,'deactivate_electrolyzer': True}
    
    return NameError(f'Unknown policy: {policy}')

In [ ]:
def is_Infeasible(state, decision, fixed_data, t):
    tol = 0.1
    if decision['power_p2h'] > fixed_data['p2h_max_rate'] + tol or decision['power_p2h'] < 0-tol:
        print(f"Policy infeasible for hour {t}: power_p2h out of bounds")
        return True
    if decision['hydrogen_h2p'] > fixed_data['h2p_max_rate'] + tol or decision['hydrogen_h2p'] < 0-tol:
        print(f"Policy infeasible for hour {t}: hydrogen_h2p out of bounds")
        return True
    if decision['power_grid'] < 0-tol or fixed_data['demand_schedule'][t] > decision['power_grid'] - decision['power_p2h'] + state['power_wind'] + fixed_data['conversion_h2p'] * decision['hydrogen_h2p'] + tol: # mini added, so it doesnt break on floating point issues
        print(f"Policy infeasible for hour {t}: power_grid out of bounds", "demand:", fixed_data['demand_schedule'][t], "supply:", decision['power_grid'], decision['power_p2h'], state['power_wind'], fixed_data['conversion_h2p'] * decision['hydrogen_h2p'])
        return True
    if decision['activate_electrolyzer'] * decision['deactivate_electrolyzer'] != 0 or (state['electrolyzer_status'] == 1 and decision['activate_electrolyzer'] == 1):
        print(f"Policy infeasible for hour {t}: activate and deactivate electrolyzer both set")
        return True
    return False

In [ ]:
def apply_dynamics(state, decision, fixed_data, t):
    p_wind = wind_model(state['power_wind'], state['power_wind_previous'], fixed_data)
    p_wind_prev = state['power_wind']
    h_tank = state['hydrogen_tank'] + decision['power_p2h'] * fixed_data['conversion_p2h'] - decision['hydrogen_h2p']
    el_price = price_model(state['el_price'], state['el_price_previous'], p_wind, fixed_data)
    el_price_prev = state['el_price']
    status = state['electrolyzer_status'] + decision['activate_electrolyzer'] - decision['deactivate_electrolyzer']
    
    return {'power_wind': p_wind, 'power_wind_previous': p_wind_prev, 'hydrogen_tank': h_tank, 'el_price': el_price, 'el_price_previous': el_price_prev, 'electrolyzer_status': status}

In [ ]:
def evaluate_daily_cost(next_state, fixed_data, policy='dummy', num_lookaheads=20, num_stages=3):
    
    results = {
    'hydrogen_tank': [],
    'power_p2h': [],
    'hydrogen_h2p': [],
    'power_grid': [],
    'electrolyzer_status': [],
    'activate_electrolyzer': [],
    'deactivate_electrolyzer': [],
    'el_price': [],
    'power_wind': [],
    'cost': [],
    'cost_total': 0
    }

    for t in range(fixed_data['num_timeslots']):
        state = next_state
        decision = apply_policy(state, fixed_data, t, policy=policy)
        infeasible = is_Infeasible(state, decision, fixed_data, t)
        if infeasible == True:
            decision = apply_policy(state, fixed_data, t, policy='dummy', num_lookaheads=num_lookaheads, num_stages=num_stages)
        next_state = apply_dynamics(state, decision, fixed_data, t)
        # Store results
        results['hydrogen_tank'].append(state['hydrogen_tank'])
        results['power_p2h'].append(decision['power_p2h'])
        results['hydrogen_h2p'].append(decision['hydrogen_h2p'])
        results['power_grid'].append(decision['power_grid'])
        results['electrolyzer_status'].append(state['electrolyzer_status'])
        results['activate_electrolyzer'].append(decision['activate_electrolyzer'])
        results['deactivate_electrolyzer'].append(decision['deactivate_electrolyzer'])
        results['el_price'].append(state['el_price'])
        results['power_wind'].append(state['power_wind'])
        results['cost'].append(state['el_price'] * decision['power_grid'] + state['electrolyzer_status'] * fixed_data['electrolyzer_cost'])
    
    results['cost_total'] = sum(results['cost'])
    return results


# Analysis

In [ ]:
fixed_data = data.get_fixed_data()

In [ ]:
wind, el_prices = get_estimates(fixed_data['num_timeslots'], fixed_data['target_mean_wind'], fixed_data['target_mean_wind'], fixed_data['mean_price'], fixed_data['mean_price'], fixed_data)
results = hindsight_model(fixed_data, el_prices, wind, fixed_data['num_timeslots'], fixed_data['initial_hydrogen'], fixed_data['initial_status'], outputFlag=0)
plot_results(results, fixed_data)
print(f"Total Cost: {sum([results['el_price'][t] * results['power_grid'][t] for t in range(fixed_data['num_timeslots'])])}")

In [ ]:
initial_state = {'power_wind': fixed_data['target_mean_wind'], 'power_wind_previous': fixed_data['target_mean_wind'], 'hydrogen_tank': fixed_data['initial_hydrogen'], 'el_price': fixed_data['mean_price'], 'el_price_previous': fixed_data['mean_price'], 'electrolyzer_status': False}
daily_cost = evaluate_daily_cost(initial_state, fixed_data, policy='two_stage', num_lookaheads=3)
for i in daily_cost:
    print(i, daily_cost[i])
plot_results(daily_cost, fixed_data=fixed_data)

In [ ]:
# Multiple runs
num_days = 10
initial_state = {'power_wind': fixed_data['target_mean_wind'], 'power_wind_previous': fixed_data['target_mean_wind'], 'hydrogen_tank': fixed_data['initial_hydrogen'], 'el_price': fixed_data['mean_price'], 'el_price_previous': fixed_data['mean_price'], 'electrolyzer_status': False}
policies = ['dummy', 'deterministic_lookahead']
all_results = {}    
for policy in policies:
    all_results[policy] = []

for policy in policies:
    for day in range(num_days):
        daily_cost = evaluate_daily_cost(initial_state, fixed_data, policy=policy)
        all_results[policy].append(daily_cost)
    print(f"Average daily cost for policy '{policy}' over {num_days} days: {sum(r['cost_total'] for r in all_results[policy])/num_days}")
    print(f"Example day ({policy}):")
    plot_results(all_results[policy][0], fixed_data)

